# 04 — Pareto MCTS Multi-Objective Optimisation

**No API key required.** Run multi-objective molecular optimisation using Monte Carlo Tree Search.

Capabilities:
- Define custom objectives (QED, MW, LogP, SA score)
- Pareto-front discovery (non-dominated solutions)
- Atom mutation + fragment addition operators
- Visualise the Pareto front

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from chemvision.generation.property_predictor import PropertyPredictor
from chemvision.generation.pareto_mcts import ParetoMCTS, Objective

pred = PropertyPredictor(use_mace=False)
print('Ready')

## 4.1 — Define objectives and run MCTS

In [ ]:
objectives = [
    Objective('QED',     fn=lambda s: pred.predict(s).qed or 0.0,                   direction='max'),
    Objective('1/MW',    fn=lambda s: 1.0 / max(pred.predict(s).mw or 500, 1),      direction='max'),
    Objective('LogP~2',  fn=lambda s: -abs((pred.predict(s).logp or 5) - 2.0),      direction='max'),
]

SEED = 'CC(=O)Oc1ccccc1C(=O)O'  # aspirin
N_ITER = 80

mcts = ParetoMCTS(objectives, max_atoms=50, seed=42)

t0 = time.perf_counter()
front = mcts.search(SEED, n_iterations=N_ITER)
elapsed = time.perf_counter() - t0

print(f'Found {len(front)} Pareto-optimal candidates in {elapsed:.1f}s')

## 4.2 — Inspect the Pareto front

In [ ]:
rows = []
for c in front[:15]:
    p = pred.predict(c.smiles)
    rows.append({
        'SMILES': c.smiles[:40] + ('...' if len(c.smiles) > 40 else ''),
        'QED': round(p.qed, 3) if p.qed else None,
        'MW': round(p.mw, 1) if p.mw else None,
        'LogP': round(p.logp, 2) if p.logp is not None else None,
        'SA': round(p.sa_score, 1) if p.sa_score else None,
        'Synth.': p.synthesisability,
        'Rank': c.pareto_rank,
    })

df = pd.DataFrame(rows)
df

## 4.3 — Visualise Pareto front

In [ ]:
qed_vals = [pred.predict(c.smiles).qed or 0 for c in front]
mw_vals = [pred.predict(c.smiles).mw or 0 for c in front]

seed_p = pred.predict(SEED)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(mw_vals, qed_vals, c='steelblue', s=60, alpha=0.7, edgecolors='navy', label='Pareto front')
ax.scatter([seed_p.mw], [seed_p.qed], c='red', s=150, marker='*', zorder=5, label=f'Seed (aspirin)')
ax.set_xlabel('Molecular Weight (g/mol)', fontsize=12)
ax.set_ylabel('QED (drug-likeness)', fontsize=12)
ax.set_title(f'Pareto Front — {len(front)} candidates from {N_ITER} MCTS iterations', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 3-objective scatter: QED vs MW, colour by LogP
logp_vals = [pred.predict(c.smiles).logp or 0 for c in front]

fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(mw_vals, qed_vals, c=logp_vals, cmap='coolwarm', s=70, edgecolors='gray')
fig.colorbar(sc, label='LogP')
ax.scatter([seed_p.mw], [seed_p.qed], c='black', s=150, marker='*', zorder=5)
ax.set_xlabel('MW (g/mol)')
ax.set_ylabel('QED')
ax.set_title('3-objective landscape: QED vs MW, coloured by LogP')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()